In [1]:
from eliminacion_de_variables import seleccion_variables_suave
import pandas as pd  # Para manipulación de datos
import matplotlib.pyplot as plt  # Para visualización
import seaborn as sns  # Para gráficos más estilizados
import numpy as np
import os

In [2]:
# Cargar los datos
csv_file_path = r"../../data/limpieza/cleaned_data.csv"
df = pd.read_csv(csv_file_path)

df.head()

,timestamp,open_binance,high_binance,low_binance,close_binance,volumen_binance,trades_binance,var_diaria_binance,rango_dia_binance,volatilidad_7_binance,...,SMA_200_dxy,EMA_200_dxy,RSI_dxy,MACD_dxy,MACD_signal_dxy,MACD_hist_dxy,pct_cambio_7d,tendencia_futura_7d,pct_cambio_30d,tendencia_futura_30d
0,2023-01-30,23743.37,23800.51,22500.00,22826.15,302405.90121,7790224.0,-3.858725,1300.51,343.123213,...,107.46170,105.562407,43.560953,-0.415354,-0.514791,0.099436,-0.278759,-1,NaN,0
1,2023-01-31,22827.38,23320.00,22714.77,23125.13,264649.34909,6798411.0,1.309814,605.23,288.912363,...,107.43190,105.527955,40.489557,-0.387432,-0.489319,0.101887,0.498722,1,NaN,0
2,2023-02-01,23125.13,23812.66,22760.23,23732.66,310790.42271,7863213.0,2.627142,1052.43,366.108175,...,107.39885,105.485090,29.528127,-0.431340,-0.477723,0.046383,-3.243041,-1,NaN,0
3,2023-02-02,23731.41,24255.00,23363.27,23488.94,364177.20751,9102300.0,-1.026939,891.73,365.258424,...,107.36960,105.447924,40.054093,-0.418546,-0.465888,0.047342,-7.205902,-1,NaN,0
4,2023-02-03,23489.33,23715.70,23204.62,23431.90,332571.02904,8007846.0,-0.242838,511.08,355.338896,...,107.34735,105.422771,55.762488,-0.310419,-0.434794,0.124375,-7.710472,-1,NaN,0


# Agrupar variables

In [3]:
sufijos = [
    "_binance",
    "_fg",
    "_gtrends",
    "_hashrate",
    "_mDbinance",
    "_sp500",
    "_nasdaq",
    "_dxy",
]

# Crear un diccionario para almacenar las columnas separadas
columnas_por_sufijo = {sufijo: [] for sufijo in sufijos}

# Recorrer las columnas y clasificarlas
for col in df.columns:
    for sufijo in sufijos:
        if col.endswith(sufijo):
            columnas_por_sufijo[sufijo].append(col)
            break

# Ahora tienes las columnas organizadas por sufijo
suma = 5 # timestamp, 4 columnas respuesta
for sufijo, columnas in columnas_por_sufijo.items():
    print(f"\nSufijo {sufijo} ({len(columnas)}):")
    print(columnas)
    suma += len(columnas)

assert suma == df.shape[1], f"Suma: {suma}, columnas:{df.shape[1]}"


Sufijo _binance (28):
['open_binance', 'high_binance', 'low_binance', 'close_binance', 'volumen_binance', 'trades_binance', 'var_diaria_binance', 'rango_dia_binance', 'volatilidad_7_binance', 'momentum_7_binance', 'volatilidad_14_binance', 'momentum_14_binance', 'volatilidad_30_binance', 'momentum_30_binance', 'SMA_7_binance', 'EMA_7_binance', 'SMA_14_binance', 'EMA_14_binance', 'SMA_30_binance', 'EMA_30_binance', 'SMA_50_binance', 'EMA_50_binance', 'SMA_200_binance', 'EMA_200_binance', 'RSI_binance', 'MACD_binance', 'MACD_signal_binance', 'MACD_hist_binance']

Sufijo _fg (1):
['fear_greed_index_fg']

Sufijo _gtrends (1):
['bitcoin_gtrends']

Sufijo _hashrate (5):
['hashrate_hashrate', 'SMA_7_hashrate', 'SMA_30_hashrate', 'EMA_7_hashrate', 'EMA_30_hashrate']

Sufijo _mDbinance (28):
['open_mDbinance', 'high_mDbinance', 'low_mDbinance', 'close_mDbinance', 'volumen_mDbinance', 'trades_mDbinance', 'var_diaria_mDbinance', 'rango_dia_mDbinance', 'volatilidad_7_mDbinance', 'momentum_7_mDbin

# Seleccion de variables suave

In [4]:
# Eliminar variables respuesta categóricas
df.drop(columns=['tendencia_futura_7d', 'tendencia_futura_30d'], inplace=True)

# Variables para 7 días
features_7d = seleccion_variables_suave(df.drop(columns=['pct_cambio_30d']), target="pct_cambio_7d")
print(f"Features para 7 días ({len(features_7d)}):\n", features_7d)
print()

df_7d = df[["timestamp"] + features_7d + ["pct_cambio_7d"]]
# Guardamos las últimas 7 filas (sin variable de respuesta) para hacer predicciones futuras
pred_7d = df_7d.tail(7).copy()
# Eliminamos las filas con valores nulos (generados por rolling y shift)
df_7d = df_7d.dropna()


# Variables para 30 días
features_30d = seleccion_variables_suave(df.drop(columns=['pct_cambio_7d']), target="pct_cambio_30d")
print(f"Features para 30 días ({len(features_30d)}):\n", features_30d)
print()

df_30d = df[["timestamp"] + features_30d + ["pct_cambio_30d"]]
# Guardamos las últimas 7 filas (sin variable de respuesta) para hacer predicciones futuras
pred_30d = df_30d.tail(30).copy()
# Eliminamos las filas con valores nulos (generados por rolling y shift)
df_30d = df_30d.dropna()

C:\Users\Daniel\Universidad\PD\c2425-R6\.venv\lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] El sistema no puede encontrar el archivo especificado
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\Daniel\Universidad\PD\c2425-R6\.venv\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Program Files\Python310\lib\subprocess.py", line 501, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Program Files\Python310\lib\subprocess.py", line 966, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Program Files\Python310\lib\subprocess.py", line 1435, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(exec

Features para 7 días (24):
 ['bitcoin_gtrends', 'open_binance', 'momentum_30_binance', 'trades_binance', 'volumen_binance', 'trades_mDbinance', 'volatilidad_7_nasdaq', 'MACD_hist_dxy', 'volume_sp500', 'volatilidad_7_sp500', 'volatilidad_30_sp500', 'volatilidad_30_nasdaq', 'EMA_200_dxy', 'volatilidad_30_dxy', 'var_diaria_nasdaq', 'SMA_50_dxy', 'momentum_30_dxy', 'volumen_mDbinance', 'rango_dia_sp500', 'MACD_hist_sp500', 'var_diaria_sp500', 'volatilidad_14_sp500', 'momentum_14_sp500', 'MACD_hist_nasdaq']

Features para 30 días (44):
 ['EMA_200_dxy', 'open_binance', 'EMA_30_dxy', 'volatilidad_30_binance', 'var_diaria_binance', 'bitcoin_gtrends', 'RSI_binance', 'MACD_hist_binance', 'momentum_30_dxy', 'volatilidad_30_sp500', 'volatilidad_14_binance', 'momentum_14_binance', 'trades_binance', 'volatilidad_30_dxy', 'MACD_hist_dxy', 'volatilidad_14_dxy', 'MACD_binance', 'momentum_30_nasdaq', 'momentum_30_sp500', 'volatilidad_14_sp500', 'rango_dia_nasdaq', 'momentum_7_binance', 'volatilidad_7_dx

In [5]:
#Guardamos los csv
output_path = r"../../data/limpieza/dataset_7d.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_7d.to_csv(output_path, index=False)

output_path = r"../../data/limpieza/pred_7d.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
pred_7d.to_csv(output_path, index=False)

In [6]:
#Guardamos los csv
output_path = r"../../data/limpieza/dataset_30d.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_30d.to_csv(output_path, index=False)

output_path = r"../../data/limpieza/pred_30d.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
pred_30d.to_csv(output_path, index=False)